## Data Reading

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
df = spark.sql("select * from databricks_cata.silver.customers_silver")
display(df)

In [0]:
init_load_flag = int(dbutils.widgets.get('init_load_flag'))
print(init_load_flag)

## Remove Duplicates

In [0]:
df = df.dropDuplicates(subset=['customer_id'])
display(df)

# ## Dividing old vs new records

In [0]:
if init_load_flag == 0:
    df_old = spark.sql(''' select DimCustomerKey, customer_id, create_date, update_date from databricks_cata.gold.DimCustomers ''')
else:
    df_old = spark.sql('''select 0 DimCustomerKey, 0 customer_id, 0 create_date, 0 update_date''')

In [0]:
display(df_old)

## Renaming Old Records

In [0]:
df_old = df_old.withColumnRenamed('DimCustomerKey','old_DimCustomerKey')\
    .withColumnRenamed('customer_id','old_customer_id')\
        .withColumnRenamed('create_date','old_create_date')\
            .withColumnRenamed('update_date','old_update_date')
display(df_old)

## Applying Joins with old records

In [0]:
df_join = df.join(df_old, df.customer_id == df_old.old_customer_id, 'left')
display(df_join)

## Separating old records and new records

In [0]:
df_new = df_join.filter(col('old_DimCustomerKey').isNull())
df_old = df_join.filter(col('old_DimCustomerKey').isNotNull())

## Preparing DF_OLD

In [0]:
df_old = df_old.drop('old_customer_id', 'old_update_date')

df_old = df_old.withColumnRenamed('old_create_date', 'create_date')
df_old = df_old.withColumnRenamed('old_DimCustomerKey', 'DimCustomerKey')

df_old = df_old.withColumn('create_date', to_timestamp('create_date')).withColumn('update_date', current_timestamp())

display(df_old)

## Preparing DF_NEW

In [0]:
df_new = df_new.drop('old_DimCustomerKey', 'old_customer_id', 'old_update_date','old_create_date')

df_new = df_new.withColumn('create_date', current_timestamp()).withColumn('update_date', current_timestamp())

In [0]:
display(df_new)

## Surrogate Key All records

In [0]:
if init_load_flag == 1:
    max_surrogant_key = 0
else:
    max_surrogant_key = df_old.select(max('DimCustomerKey')).collect()[0][0]

print(max_surrogant_key)

In [0]:
df_new = df_new.withColumn('DimCustomerKey', lit(max_surrogant_key) + monotonically_increasing_id()+lit(1))
display(df_new)

## Union All DFs

In [0]:
df_final = df_new.unionByName(df_old)
display(df_final)

## Upsert Data

In [0]:
from delta.tables import DeltaTable

In [0]:
if init_load_flag == 1:
    print('in')
    df_final.write.mode('overwrite').format('delta').option('path','abfss://gold@databrickse2esa.dfs.core.windows.net/DimCustomers')\
        .saveAsTable("databricks_cata.gold.DimCustomers")
    dbutils.widgets.text('init_load_flag', '0')
else:
    print('else')
    target = DeltaTable.forPath(spark, 'abfss://gold@databrickse2esa.dfs.core.windows.net/DimCustomers')

    target.alias('trg').merge(df_final.alias('src'), "trg.DimCustomerKey = src.DimCustomerKey")\
        .whenMatchedUpdateAll()\
            .whenNotMatchedInsertAll()\
                .execute()

display(spark.read.table('databricks_cata.gold.DimCustomers'))